In [1]:
import pandas as pd

players = pd.read_csv("../data/players.csv", engine="python")
players["has_value"] = players["market_value_in_eur"].notnull()

In [2]:
pd.crosstab(players["last_season"], players["has_value"]).tail(8)

has_value,False,True
last_season,,
2018,82,1983
2019,67,1499
2020,86,1757
2021,130,1687
2022,143,1734
2023,238,1854
2024,3572,2503
2025,3888,18404


In [3]:
pd.crosstab(
    players["current_club_domestic_competition_id"].isnull(),
    players["has_value"],
)

has_value,False,True
current_club_domestic_competition_id,,
False,7829,39334
True,792,2194


In [4]:
players["international_caps"].describe()
(players["international_caps"] == 0).sum()

np.int64(2410)

In [5]:
active = players[players["last_season"] == 2025].copy()
print(active.shape)
print(active["market_value_in_eur"].notnull().sum())

(22292, 27)
18404


In [6]:
data = active.dropna(subset=["market_value_in_eur"]).copy()
print(data.shape)
data["current_club_domestic_competition_id"].isnull().sum()

(18404, 27)


np.int64(2194)

In [7]:
print(data["international_caps"].isnull().sum())
print((data["international_caps"] == 0).sum())

4496
1545


In [8]:
caps_group = pd.cut(
    data["international_caps"].fillna(-1),
    bins=[-2, -0.5, 0.5, 1000],
    labels=["missing", "zero", "has caps"],
)
data.groupby(caps_group, observed=True)["market_value_in_eur"].agg(["count", "median"])

,count,median
international_caps,,
missing,4496,400000.0
zero,1545,500000.0
has caps,12363,700000.0


In [9]:
data["caps_missing"] = data["international_caps"].isnull().astype(int)
data["international_caps"] = data["international_caps"].fillna(0)
data["international_goals"] = data["international_goals"].fillna(0)
data["caps_missing"].sum()

np.int64(4496)

In [10]:
data["date_of_birth"] = pd.to_datetime(data["date_of_birth"])
ref_date = pd.Timestamp("2026-06-12")
data["age"] = (ref_date - data["date_of_birth"]).dt.days / 365.25
data["age"].describe()

count    18404.000000
mean        26.552248
std          4.743231
min         16.177960
25%         22.877481
50%         26.119097
75%         29.755647
max         45.697467
Name: age, dtype: float64

In [11]:
cols = ["height_in_cm", "foot", "contract_expiration_date", "sub_position", "date_of_birth"]
data[cols].isnull().sum()

height_in_cm                 705
foot                        1404
contract_expiration_date    3434
sub_position                  29
date_of_birth                  0
dtype: int64

In [12]:
data.groupby(data["contract_expiration_date"].isnull())["market_value_in_eur"].agg(["count", "median"])

,count,median
contract_expiration_date,,
False,14970,700000.0
True,3434,300000.0


In [13]:
data.groupby(data["current_club_domestic_competition_id"].isnull())["market_value_in_eur"].agg(["count", "median"])

,count,median
current_club_domestic_competition_id,,
False,16210,700000.0
True,2194,275000.0


In [14]:
no_league = data[data["current_club_domestic_competition_id"].isnull()]
no_league[["name", "current_club_id", "current_club_name", "contract_expiration_date"]].head(10)

,name,current_club_id,current_club_name,contract_expiration_date
134,Paolo Guerrero,184,NaN,2026-12-31 00:00:00
2502,Guillermo Ochoa,786,NaN,2026-05-31 00:00:00
3204,Arturo Vidal,2433,NaN,2026-12-31 00:00:00
3642,Taijo Teniste,28783,NaN,2026-12-31 00:00:00
3643,Konstantin Vassiljev,123,NaN,NaN
3669,Grant Kekana,6356,NaN,2026-06-30 00:00:00
3779,Marc Pujol,64780,NaN,2026-06-30 00:00:00
3784,Sergi Moreno,70956,NaN,2026-06-30 00:00:00
4072,Stevan Jovetić,829,NaN,2026-05-31 00:00:00
4137,Teemu Pukki,1008,NaN,2026-12-31 00:00:00


In [15]:
data["current_club_domestic_competition_id"].nunique()

32

In [16]:
data["contract_months_left"] = (
    pd.to_datetime(data["contract_expiration_date"]) - ref_date
).dt.days / 30.44
(data["contract_months_left"] < 0).sum()

np.int64(290)

In [17]:
data["height_in_cm"] = data["height_in_cm"].fillna(
    data.groupby("position")["height_in_cm"].transform("median")
)

In [18]:
data["foot"] = data["foot"].fillna("unknown")
data["sub_position"] = data["sub_position"].fillna("Unknown")

In [19]:
data["no_league"] = data["current_club_domestic_competition_id"].isnull().astype(int)
data["current_club_domestic_competition_id"] = data["current_club_domestic_competition_id"].fillna("UNKNOWN")

In [20]:
data["has_contract_info"] = data["contract_expiration_date"].notnull().astype(int)
data["contract_months_left"] = data["contract_months_left"].clip(lower=0).fillna(0)

In [21]:
data[["height_in_cm", "foot", "sub_position", "current_club_domestic_competition_id", "contract_months_left"]].isnull().sum()

height_in_cm                            0
foot                                    0
sub_position                            0
current_club_domestic_competition_id    0
contract_months_left                    0
dtype: int64

In [22]:
exp = pd.to_datetime(data["contract_expiration_date"])
data[exp < ref_date]["contract_expiration_date"].value_counts().head(5)

contract_expiration_date
2026-01-31 00:00:00    171
2026-05-31 00:00:00    108
2026-05-30 00:00:00      4
2026-04-30 00:00:00      2
2026-02-20 00:00:00      1
Name: count, dtype: int64

In [23]:
drop_cols = [
    "first_name", "last_name", "last_season", "current_club_id",
    "player_code", "country_of_birth", "city_of_birth", "date_of_birth",
    "agent_name", "image_url", "url", "current_national_team_id",
    "current_club_name", "contract_expiration_date",
    "highest_market_value_in_eur", "has_value",
]
model_df = data.drop(columns=drop_cols)
print(model_df.shape)
model_df.isnull().sum()

(18404, 16)


player_id                               0
name                                    0
country_of_citizenship                  0
sub_position                            0
position                                0
foot                                    0
height_in_cm                            0
international_caps                      0
international_goals                     0
current_club_domestic_competition_id    0
market_value_in_eur                     0
caps_missing                            0
age                                     0
contract_months_left                    0
no_league                               0
has_contract_info                       0
dtype: int64

In [24]:
model_df.to_csv("../data/players_clean.csv", index=False)